# 🧬 Unsupervised Epigenetic Aging Analysis on Macaque Liver RRBS Data

This notebook implements a complete pipeline to analyze DNA methylation levels from 94 rhesus macaques (*Macaca mulatta*) aged 3 to 40. 
Using global methylation statistics—**Mean**, **Median**, and **Variability** (standard deviation)—we build an unsupervised age-proxy model, assign broad age categories, rank individuals by relative health within their cohort, and project the feature space using **UMAP**.

## 1. Feature Extraction & Epigenetic Loci Correlation
We load the high-dimensional RData file and extract three global statistics for each of the 94 samples across all 235,810 CpG sites. We also perform an **Epigenome-wide Association Study (EWAS)** by calculating the Pearson correlation coefficient between each individual CpG site's methylation levels and the global Aging Score vector. This allows us to identify the individual CpG sites that are most predictive of biological aging.

In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np
import umap
import plotly.express as px
import plotly.graph_objects as go

# Define paths
rdata_path = '/Users/hellyr/Documents/ONPRCliverSitePercMeth5x1090n94.RData'
features_csv = 'macaque_features.csv'
correlations_csv = 'macaque_top_correlations.csv'

r_code = f"""
load('{rdata_path}')

# Extract global features
mean_vals <- colMeans(perc5X, na.rm=TRUE)
median_vals <- apply(perc5X, 2, median, na.rm=TRUE)
sd_vals <- apply(perc5X, 2, sd, na.rm=TRUE)

df <- data.frame(
  sample = colnames(perc5X),
  mean = mean_vals,
  median = median_vals,
  variability = sd_vals
)

# Standardize features to compute Aging Score
mean_std <- (df$mean - mean(df$mean)) / sd(df$mean)
median_std <- (df$median - mean(df$median)) / sd(df$median)
sd_std <- (df$variability - mean(df$variability)) / sd(df$variability)
aging_score <- sd_std - mean_std - median_std

# Compute Pearson correlation for each CpG site with Aging Score
print('Computing CpG correlations (EWAS)...')
cor_vals <- cor(t(perc5X), aging_score, use = 'pairwise.complete.obs')
cor_df <- data.frame(
  cpg = rownames(cor_vals),
  correlation = cor_vals[, 1]
)
cor_df <- cor_df[!is.na(cor_df$correlation), ]

# Get top 10 hypermethylated and top 10 hypomethylated CpG sites
cor_df_sorted <- cor_df[order(cor_df$correlation, decreasing = TRUE), ]
top_pos <- head(cor_df_sorted, 10)
top_neg <- tail(cor_df_sorted, 10)
top_cors <- rbind(top_pos, top_neg)

write.csv(df, '{features_csv}', row.names=FALSE)
write.csv(top_cors, '{correlations_csv}', row.names=FALSE)
print('Features and top correlations successfully exported!')
"""

# Run feature extraction 
with open('extract_features.R', 'w') as f:
    f.write(r_code)

print("Running feature extraction in R...")
result = subprocess.run(['Rscript', 'extract_features.R'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Error in R execution:", result.stderr)
    raise RuntimeError(f"R script failed: {{result.stderr}}")

# Clean up
if os.path.exists('extract_features.R'):
    os.remove('extract_features.R')

# Load extracted features into pandas
df = pd.read_csv(features_csv)
df.head()

Running feature extraction in R...


[1] "Computing CpG correlations (EWAS)..."
[1] "Features and top correlations successfully exported!"



,sample,mean,median,variability
0,ECL50_1_S1,0.632316,0.714286,0.300796
1,ECL50_10_S10,0.621295,0.700000,0.273830
2,ECL50_11_S11,0.644813,0.727273,0.258908
3,ECL50_12_S12,0.648780,0.727273,0.262725
4,ECL50_13_S13,0.652972,0.727273,0.250722


## 2. Epigenetic Age Imputation (Aging Score)
We calculate the standardized $z$-scores for the three features. Since mean/median methylation decrease during biological aging and cell-to-cell variability increases, we define the **Aging Score** as:
$$\text{Aging Score} = \text{Variability}_{std} - \text{Mean}_{std} - \text{Median}_{std}$$

We then rank the samples based on this score and map them to the chronological age range of the cohort (3 to 40 years) using a uniform distribution mapping.

In [2]:
# Standardize features
df['mean_std'] = (df['mean'] - df['mean'].mean()) / df['mean'].std()
df['median_std'] = (df['median'] - df['median'].mean()) / df['median'].std()
df['variability_std'] = (df['variability'] - df['variability'].mean()) / df['variability'].std()

# Calculate Aging Score
df['aging_score'] = df['variability_std'] - df['mean_std'] - df['median_std']

# Map ranks to chronological age [3, 40]
df = df.sort_values('aging_score').reset_index(drop=True)
N = len(df)
df['estimated_age'] = 3 + (40 - 3) * (df.index) / (N - 1)
df.head()

,sample,mean,median,variability,mean_std,median_std,variability_std,aging_score,estimated_age
0,ECL50_2_S2,0.658941,0.736842,0.265253,2.137855,1.724472,-0.945958,-4.808285,3.000000
1,ECL50_13_S13,0.652972,0.727273,0.250722,1.820185,1.317900,-1.615610,-4.753695,3.397849
2,ECL50_9_S9,0.650746,0.727273,0.254033,1.701768,1.317900,-1.462992,-4.482660,3.795699
3,ECL50_12_S12,0.648780,0.727273,0.262725,1.597127,1.317900,-1.062434,-3.977461,4.193548
4,ECL50_11_S11,0.644813,0.727273,0.258908,1.386007,1.317900,-1.238330,-3.942237,4.591398


## 3. Categorization & Relative Health Ranking
We group the estimated ages into broad categories:
- **Juvenile** (3-5 years)
- **Young Adult** (6-8 years)
- **Adult** (8-15 years)
- **Old** (> 15 years)

Within each category, we rank individuals by **Relative Health** based on their Aging Score (lower score = healthier / epigenetically younger).

In [3]:
def get_age_category(age):
    if age <= 5:
        return 'Juvenile (3-5)'
    elif age <= 8:
        return 'Young Adult (6-8)'
    elif age <= 15:
        return 'Adult (8-15)'
    else:
        return 'Old (> 15)'

df['age_category'] = df['estimated_age'].apply(get_age_category)

# Health Rank within category (lower aging score is healthier)
df['relative_health_rank'] = df.groupby('age_category')['aging_score'].rank(method='first', ascending=True).astype(int)

# Apply ordered categorical type for plotting
category_order = ['Juvenile (3-5)', 'Young Adult (6-8)', 'Adult (8-15)', 'Old (> 15)']
df['age_category'] = pd.Categorical(df['age_category'], categories=category_order, ordered=True)
df_sorted = df.sort_values(['age_category', 'relative_health_rank']).reset_index(drop=True)
df_sorted.to_csv('macaque_analysis_results.csv', index=False)
df_sorted[['sample', 'estimated_age', 'age_category', 'relative_health_rank']].head(10)

,sample,estimated_age,age_category,relative_health_rank
0,ECL50_2_S2,3.000000,Juvenile (3-5),1
1,ECL50_13_S13,3.397849,Juvenile (3-5),2
2,ECL50_9_S9,3.795699,Juvenile (3-5),3
3,ECL50_12_S12,4.193548,Juvenile (3-5),4
4,ECL50_11_S11,4.591398,Juvenile (3-5),5
5,ECL50_60_S60,4.989247,Juvenile (3-5),6
6,ECL50_43_S43,5.387097,Young Adult (6-8),1
7,ECL50_47_S47,5.784946,Young Adult (6-8),2
8,ECL50_35_S35,6.182796,Young Adult (6-8),3
9,ECL50_4_S4,6.580645,Young Adult (6-8),4


## 4. Dimensionality Reduction (UMAP)
We use UMAP to project the 3-dimensional summary statistics feature space `[Mean, Median, Variability]` into 2 dimensions for visualization.

In [4]:
# Scale the features for UMAP
features = df_sorted[['mean', 'median', 'variability']]
features_scaled = (features - features.mean()) / features.std()

# Fit UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(features_scaled)

df_sorted['umap_1'] = embedding[:, 0]
df_sorted['umap_2'] = embedding[:, 1]
df_sorted.head()

/opt/homebrew/anaconda3/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,sample,mean,median,variability,mean_std,median_std,variability_std,aging_score,estimated_age,age_category,relative_health_rank,umap_1,umap_2
0,ECL50_2_S2,0.658941,0.736842,0.265253,2.137855,1.724472,-0.945958,-4.808285,3.000000,Juvenile (3-5),1,7.945685,-3.822979
1,ECL50_13_S13,0.652972,0.727273,0.250722,1.820185,1.317900,-1.615610,-4.753695,3.397849,Juvenile (3-5),2,7.617919,-4.069261
2,ECL50_9_S9,0.650746,0.727273,0.254033,1.701768,1.317900,-1.462992,-4.482660,3.795699,Juvenile (3-5),3,7.782455,-3.858452
3,ECL50_12_S12,0.648780,0.727273,0.262725,1.597127,1.317900,-1.062434,-3.977461,4.193548,Juvenile (3-5),4,7.912477,-3.948412
4,ECL50_11_S11,0.644813,0.727273,0.258908,1.386007,1.317900,-1.238330,-3.942237,4.591398,Juvenile (3-5),5,7.710215,-4.026952


## 5. Interactive UMAP Visualization
We create a premium, interactive Plotly scatter plot showing the UMAP projection of the samples. Hovering over a point displays its epigenetic profile, estimated age, cohort category, and health ranking.

In [5]:
fig = px.scatter(
    df_sorted,
    x='umap_1',
    y='umap_2',
    color='age_category',
    title='UMAP Projection of Macaque Epigenetic Aging Profiles',
    labels={'umap_1': 'UMAP 1', 'umap_2': 'UMAP 2', 'age_category': 'Age Category'},
    hover_name='sample',
    hover_data={
        'umap_1': False,
        'umap_2': False,
        'estimated_age': ':.2f',
        'relative_health_rank': True,
        'mean': ':.4f',
        'median': ':.4f',
        'variability': ':.4f'
    },
    color_discrete_sequence=px.colors.qualitative.Prism
)

fig.update_traces(
    marker=dict(size=12, opacity=0.85, line=dict(width=1, color='DarkSlateGrey')),
    selector=dict(mode='markers')
)

fig.update_layout(
    template='plotly_dark',
    title_font=dict(size=20, family='Outfit, sans-serif'),
    legend_title_font=dict(size=14),
    legend_font=dict(size=12),
    xaxis=dict(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)', zeroline=False),
    paper_bgcolor='rgba(15, 15, 15, 1)',
    plot_bgcolor='rgba(15, 15, 15, 1)',
    margin=dict(l=40, r=40, t=60, b=40),
    width=900,
    height=600
)

fig.show()

## 6. Annotation of Most Predictive Epigenetic Loci
We load the Pearson correlation coefficients computed in Step 1 to visualize the top 20 individual CpG features (top 10 hypermethylated and top 10 hypomethylated) that correlate most strongly with the epigenetic aging score.

In [ ]:
# Load top correlations
cor_df = pd.read_csv(correlations_csv)
cor_df = cor_df.sort_values('correlation', ascending=True)

# Create a category color column
cor_df['type'] = np.where(cor_df['correlation'] > 0, 'Hypermethylated (Age-gain)', 'Hypomethylated (Age-loss)')

fig_bar = px.bar(
    cor_df,
    x='correlation',
    y='cpg',
    color='type',
    orientation='h',
    title='Top 20 CpG Sites Most Predictive of Epigenetic Aging',
    labels={'correlation': 'Pearson Correlation with Aging Score', 'cpg': 'CpG Site Locus'},
    color_discrete_map={
        'Hypermethylated (Age-gain)': '#E06666',
        'Hypomethylated (Age-loss)': '#6FA8DC'
    }
)

fig_bar.update_layout(
    template='plotly_dark',
    title_font=dict(size=18, family='Outfit, sans-serif'),
    xaxis=dict(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)', range=[-1.05, 1.05]),
    yaxis=dict(showgrid=False, title_font=dict(size=14)),
    paper_bgcolor='rgba(15, 15, 15, 1)',
    plot_bgcolor='rgba(15, 15, 15, 1)',
    margin=dict(l=120, r=40, t=60, b=40),
    width=900,
    height=600,
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center')
)

fig_bar.show()

## 7. Cohort Summary & Top Health Rankings
Here we show the top 3 healthiest (epigenetically youngest) macaques in each of the 4 age categories.

In [ ]:
top_health = df_sorted[df_sorted['relative_health_rank'] <= 3]
top_health_summary = top_health[['age_category', 'relative_health_rank', 'sample', 'estimated_age', 'mean', 'variability']]
top_health_summary = top_health_summary.sort_values(['age_category', 'relative_health_rank'])
top_health_summary

: 